## 1. Set keywords to search for

Translate english keywords into Japanese to search for in the manifestos

Deflation = デフレーション, デフレ \
Nuclear Power = 原子力 \
China = 中国 \
Constitution = 憲法 \
Migration = 移民 (Uncommon), 外国人労働者 ("Foreign Worker", More Common)

In [2]:
import pandas as pd
import spacy
from collections import Counter

nlp = spacy.load("ja_core_news_sm")

# Make sure text length isnt too long for spacy
def chunk_text(text, max_bytes=40000):
    encoded = text.encode("utf-8")
    chunks = []
    while encoded:
        chunk = encoded[:max_bytes]
        # avoid splitting mid-character
        chunk = chunk.decode("utf-8", errors="ignore")
        chunks.append(chunk)
        encoded = encoded[len(chunk.encode("utf-8")):]
    return chunks

# Exclude common particles etc.
excluded_words = {"の", "に", "は", "を", "た", "が", "で", "て", "と", "し", "ます", "する", "な", "や", "も", "へ"}

# Tokenize manifesto text
def tokenize(csv_path):
    df = pd.read_csv(csv_path)
    all_tokens = []
    
    for text in df["text"]:
        for chunk in chunk_text(str(text)):
            doc = nlp(chunk)
            tokens = [token.text for token in doc 
                      if not token.is_space 
                      and not token.is_punct
                      and token.text not in excluded_words]
            all_tokens.extend(tokens)
    
    token_df = pd.DataFrame(all_tokens, columns=["word"])
    word_counts_df = token_df["word"].value_counts().reset_index()
    word_counts_df.columns = ["word", "count"]
    
    return word_counts_df

word_counts_df = tokenize("../../project/data/clean/Japan_sorted_manifestos.csv")

In [3]:
word_counts_df.head(10)

,word,count
0,的,566
1,化,530
2,者,444
3,等,409
4,法,370
5,支援,364
6,が,350
7,ため,331
8,で,314
9,など,313


In [4]:
# Search for count of a specific word
def wordsearch(keyword):
    count = word_counts_df.loc[word_counts_df['word'] == keyword, 'count'].item()
    return count

wordsearch('的')

566

In [7]:
import re

phrases = ["経済の再生", "新しい資本主義", "物価高対策", "全世代型社会保障",
           "政治とカネ", "国益を守る", "日本第一"]

def phrase_counts(csv_path, phrases):
    df = pd.read_csv(csv_path)
    df["text"] = df["text"].astype(str)

    results = []
    for phrase in phrases:
        df["count"] = df["text"].str.count(re.escape(phrase))
        total = df["count"].sum()
        by_party = df.groupby("partyname")["count"].sum()
        results.append({
            "phrase": phrase,
            "total": total,
            **by_party.to_dict()
        })

    return pd.DataFrame(results).fillna(0)

results_df = phrase_counts("../../project/data/clean/Japan_sorted_manifestos.csv", phrases)
results_df

,phrase,total,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party
0,経済の再生,1,0,0,1,0,0,0,0
1,新しい資本主義,0,0,0,0,0,0,0,0
2,物価高対策,0,0,0,0,0,0,0,0
3,全世代型社会保障,2,0,0,0,2,0,0,0
4,政治とカネ,5,0,1,3,0,0,0,1
5,国益を守る,1,0,0,0,1,0,0,0
6,日本第一,0,0,0,0,0,0,0,0


In [14]:
phrases = ["デフレ", "デフレーション", "原子力", "憲法", "移民", "外国人労働者","国家安全保障", "安全保障"]

def phrase_counts_by_date(csv_path, phrases, start_year=2001, end_year=2021):
    df = pd.read_csv(csv_path)
    df["text"] = df["text"].astype(str)

    df["year"] = pd.to_datetime(df["date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    results = []
    for phrase in phrases:
        df["count"] = df["text"].str.count(re.escape(phrase))
        total = df["count"].sum()
        by_party = df.groupby("partyname")["count"].sum()
        results.append({
            "phrase": phrase,
            "total": total,
            **by_party.to_dict()
        })

    return pd.DataFrame(results).fillna(0)

results_df = phrase_counts_by_date(
    "../../project/data/clean/Japan_sorted_manifestos.csv",
    phrases,
    start_year=2001,
    end_year=2021
)
results_df

,phrase,total,Constitutional Democratic Party of Japan,Japan Restoration Party,Japanese Communist Party,Liberal Democratic Party,New Clean Government Party,Party of Hope,Social Democratic Party
0,デフレ,1,0,0,0,0,0,0,1
1,デフレーション,0,0,0,0,0,0,0,0
2,原子力,23,0,5,1,6,1,3,7
3,憲法,153,7,20,49,13,13,20,31
4,移民,0,0,0,0,0,0,0,0
5,外国人労働者,2,0,0,0,1,0,1,0
6,国家安全保障,0,0,0,0,0,0,0,0
7,安全保障,43,3,12,2,3,8,9,6


In [17]:
from collections import Counter

def top_words_by_party(csv_path, top_n=10):
    df = pd.read_csv(csv_path)
    results = {}

    for party, group in df.groupby("partyname"):
        tokens = []
        for text in group["text"]:
            for chunk in chunk_text(str(text)):
                doc = nlp(chunk)
                tokens.extend([
                    token.text for token in doc
                    if not token.is_space
                    and not token.is_punct
                    and token.text not in excluded_words
                ])
        results[party] = Counter(tokens).most_common(top_n)

    return results

party_top_words = top_words_by_party("../../project/data/clean/Japan_sorted_manifestos.csv", top_n=10)

for party, words in party_top_words.items():
    print(f"\n{party}:")
    for word, count in words:
        print(f"  {word}: {count}")



Constitutional Democratic Party of Japan:
  社会: 17
  的: 13
  さ: 12
  実現: 12
  原発: 12
  です: 11
  法: 11
  支援: 10
  化: 10
  主義: 9

Japan Restoration Party:
  法: 134
  化: 123
  案: 112
  平成: 107
  改革: 100
  等: 83
  年: 72
  で: 71
  大阪: 63
  的: 60

Japanese Communist Party:
  日本: 173
  です: 164
  い: 152
  税: 149
  こと: 143
  企業: 139
  党: 136
  など: 136
  的: 131
  円: 131

Liberal Democratic Party:
  的: 125
  化: 120
  等: 117
  で: 103
  が: 95
  る: 89
  など: 84
  者: 81
  国: 77
  す: 75

New Clean Government Party:
  等: 161
  など: 155
  が: 147
  支援: 147
  化: 136
  者: 126
  的: 122
  ため: 116
  推進: 109
  で: 81

Party of Hope:
  で: 59
  が: 57
  など: 53
  ゼロ: 36
  より: 32
  社会: 30
  こと: 28
  化: 28
  的: 27
  国民: 21

Social Democratic Party:
  的: 88
  法: 84
  者: 79
  ○: 70
  支援: 70
  など: 67
  地域: 65
  制度: 61
  化: 55
  強化: 49
